# Adım 6 — Makine Öğrenmesi: Çoklu Model KarşılaştırmaBu notebook, **5 farklı sınıflandırma modelinin** eğitim sonuçlarını MLflow'dan yükleyerek karşılaştırır.## Eğitilen Modeller| # | Model | Script | Avantaj ||---|-------|--------|---------|| 1 | **Logistic Regression** | `ml/01_logistic_regression.py` | Hızlı, yorumlanabilir katsayılar || 2 | **Decision Tree** | `ml/02_decision_tree.py` | Yüksek yorumlanabilirlik, feature importance || 3 | **Random Forest** | `ml/03_random_forest.py` | Ensemble — düşük overfitting riski || 4 | **Gradient Boosted Trees** | `ml/04_gbt.py` | En güçlü aday, ardışık hata düzeltme || 5 | **Naive Bayes** | `ml/05_naive_bayes.py` | Hızlı baseline, MinMaxScaler gerekli |## Değerlendirme Metrikleri- **AUC-ROC** — Sınıf ayrım gücü- **Accuracy** — Genel doğruluk- **F1-Score** — Precision-Recall dengesi- **Precision** — Yanlış pozitif oranı- **Recall** — Kaçırılan saldırı oranı- **Confusion Matrix** — TP, TN, FP, FN detayı

In [ ]:
# MLflow sonuçlarını yükleimport sqlite3import pandas as pdimport osfrom pathlib import PathPROJECT_ROOT = Path(os.pardir).resolve()MLFLOW_DB = PROJECT_ROOT / "mlruns" / "mlflow.db"if not MLFLOW_DB.exists():    print("⚠️ MLflow veritabanı bulunamadı. Modelleri eğittikten sonra tekrar çalıştırın.")    print(f"   Beklenen konum: {MLFLOW_DB}")else:    con = sqlite3.connect(str(MLFLOW_DB))    runs = pd.read_sql_query(        """SELECT run_uuid, name, status, start_time, end_time           FROM runs WHERE status = 'FINISHED'           ORDER BY start_time DESC""", con)    metrics = pd.read_sql_query("SELECT run_uuid, key, value FROM metrics", con)    tags = pd.read_sql_query(        """SELECT run_uuid, key, value FROM tags           WHERE key IN ('model_type', 'classification_type')""", con)    con.close()    print(f"Toplam tamamlanmış run: {len(runs)}")    print(f"Toplam metrik kaydı: {len(metrics)}")    if not tags.empty:        model_types = tags[tags["key"] == "model_type"]["value"].unique()        print(f"Eğitilen model türleri: {list(model_types)}")

In [ ]:
# Her model için en iyi run'ı bul ve karşılaştırma tablosu oluşturif MLFLOW_DB.exists() and not runs.empty:    # Tag'leri ve metrikleri pivotla    tag_pivot = tags.pivot_table(index="run_uuid", columns="key", values="value", aggfunc="first").reset_index()    metric_pivot = (metrics.groupby(["run_uuid", "key"], as_index=False)                    .last()                    .pivot(index="run_uuid", columns="key", values="value")                    .reset_index())    df = runs.merge(tag_pivot, on="run_uuid", how="left").merge(metric_pivot, on="run_uuid", how="left")    # Model türüne göre en iyi accuracy    if "model_type" in df.columns and "accuracy" in df.columns:        best = df.sort_values("accuracy", ascending=False).drop_duplicates(subset=["model_type"])        display_cols = ["model_type", "accuracy", "f1_score", "precision", "recall", "auc_roc"]        display_cols = [c for c in display_cols if c in best.columns]        result = best[display_cols].copy()        for c in result.columns:            if c != "model_type":                result[c] = result[c].apply(lambda v: f"{v:.4f}" if pd.notna(v) else "—")        print("\n📊 Model Karşılaştırma Tablosu")        print("═" * 80)        print(result.to_string(index=False))    else:        print("model_type tag'i veya accuracy metriği bulunamadı.")else:    print("MLflow verisi yok — önce modelleri eğitin.")

In [ ]:
# Grouped Bar Chart — 5 Modelin Metrik Karşılaştırmasıtry:    import matplotlib.pyplot as plt    import numpy as np    if MLFLOW_DB.exists() and "model_type" in df.columns and "accuracy" in df.columns:        best = df.sort_values("accuracy", ascending=False).drop_duplicates(subset=["model_type"])        metrics_list = ["accuracy", "f1_score", "precision", "recall"]        metrics_list = [m for m in metrics_list if m in best.columns]        models = best["model_type"].tolist()        x = np.arange(len(models))        width = 0.18        fig, ax = plt.subplots(figsize=(14, 6))        colors = ["#6366F1", "#10B981", "#F59E0B", "#EF4444"]        for i, metric in enumerate(metrics_list):            values = best[metric].astype(float).tolist()            ax.bar(x + i * width, values, width, label=metric, color=colors[i % len(colors)])        ax.set_xlabel("Model")        ax.set_ylabel("Skor")        ax.set_title("5 Modelin Performans Karşılaştırması (Grouped Bar Chart)")        ax.set_xticks(x + width * 1.5)        ax.set_xticklabels(models, rotation=15, ha="right")        ax.set_ylim(0, 1.05)        ax.legend()        ax.grid(axis="y", alpha=0.3)        plt.tight_layout()        plt.show()    else:        print("Grafik için yeterli veri yok.")except ImportError:    print("matplotlib yüklü değil — pip install matplotlib")

In [ ]:
# Feature Importance — En iyi modeldenML_DIR = PROJECT_ROOT / "ml"fi_files = {    "Decision Tree": "dt_feature_importance.csv",    "Random Forest": "rf_feature_importance.csv",    "GBT": "gbt_feature_importance.csv",    "Naive Bayes": "nb_feature_importance.csv",}best_fi = Nonebest_name = Nonefor name, fname in fi_files.items():    path = ML_DIR / fname    if path.exists():        fi = pd.read_csv(path).sort_values("importance", ascending=False)        if best_fi is None or fi["importance"].max() > best_fi["importance"].max():            best_fi = fi            best_name = nameif best_fi is not None:    print(f"Feature Importance — {best_name} (Top 15)")    print("═" * 50)    top15 = best_fi.head(15)    for _, row in top15.iterrows():        bar = "█" * int(row["importance"] * 50)        print(f"  {row['feature_name']:<30} {row['importance']:.4f}  {bar}")    # Türetilmiş feature'ları vurgula    eng = {"traffic_asymmetry_ratio", "pkt_size_cv", "flow_intensity", "iat_regularity", "conn_efficiency"}    found_eng = top15[top15["feature_name"].isin(eng)]    if not found_eng.empty:        print(f"\n🎯 Top 15 içinde {len(found_eng)} türetilmiş feature: {list(found_eng['feature_name'])}")else:    print("Feature importance CSV dosyası bulunamadı — modelleri eğitin.")

In [ ]:
# Feature Importance — Horizontal Bar Charttry:    if best_fi is not None:        top15 = best_fi.head(15).iloc[::-1]  # ters sıra (en önemli üstte)        fig, ax = plt.subplots(figsize=(10, 7))        colors = ["#EC4899" if f in eng else "#6366F1" for f in top15["feature_name"]]        ax.barh(top15["feature_name"], top15["importance"], color=colors)        ax.set_xlabel("Önem Skoru")        ax.set_title(f"Feature Importance — {best_name} (Top 15)")        # Lejand        from matplotlib.patches import Patch        legend_elements = [            Patch(facecolor="#EC4899", label="Türetilmiş feature"),            Patch(facecolor="#6366F1", label="Orijinal feature"),        ]        ax.legend(handles=legend_elements, loc="lower right")        ax.grid(axis="x", alpha=0.3)        plt.tight_layout()        plt.show()except Exception as e:    print(f"Grafik hatası: {e}")

In [ ]:
# Confusion Matrix — En iyi modelif MLFLOW_DB.exists() and not runs.empty:    # En iyi accuracy run    if "accuracy" in df.columns:        top_run = df.sort_values("accuracy", ascending=False).iloc[0]        run_id = top_run["run_uuid"]        con = sqlite3.connect(str(MLFLOW_DB))        cm_metrics = pd.read_sql_query(            "SELECT key, value FROM metrics WHERE run_uuid = ? AND key LIKE 'cm_%'",            con, params=(run_id,))        con.close()        if not cm_metrics.empty:            cm_dict = dict(zip(cm_metrics["key"], cm_metrics["value"]))            tp = cm_dict.get("cm_TP", 0)            tn = cm_dict.get("cm_TN", 0)            fp = cm_dict.get("cm_FP", 0)            fn = cm_dict.get("cm_FN", 0)            print(f"Confusion Matrix — {top_run.get('model_type', top_run['name'])}")            print("═" * 40)            print(f"                  Predicted")            print(f"                  Normal  Saldırı")            print(f"  Actual Normal   {int(tn):>6}  {int(fp):>6}")            print(f"  Actual Saldırı  {int(fn):>6}  {int(tp):>6}")            print(f"\n  Toplam: {int(tn+fp+fn+tp):,}")            if (tp+fn) > 0:                print(f"  Recall (Saldırı):  {tp/(tp+fn)*100:.2f}%")            if (tp+fp) > 0:                print(f"  Precision (Saldırı): {tp/(tp+fp)*100:.2f}%")            # Heatmap            try:                import numpy as np                cm = np.array([[tn, fp], [fn, tp]])                fig, ax = plt.subplots(figsize=(6, 5))                im = ax.imshow(cm, cmap="YlOrRd")                ax.set_xticks([0, 1])                ax.set_yticks([0, 1])                ax.set_xticklabels(["Normal (0)", "Saldırı (1)"])                ax.set_yticklabels(["Normal (0)", "Saldırı (1)"])                ax.set_xlabel("Tahmin")                ax.set_ylabel("Gerçek")                ax.set_title("Confusion Matrix")                for i in range(2):                    for j in range(2):                        ax.text(j, i, f"{int(cm[i,j]):,}", ha="center", va="center", fontsize=14, fontweight="bold")                plt.colorbar(im)                plt.tight_layout()                plt.show()            except Exception:                pass        else:            print("Bu run için confusion matrix metriği bulunamadı.")

## Model Eğitim Komutları```bash# 1) Logistic Regressiondocker exec spark-master spark-submit --driver-memory 2g --packages io.delta:delta-core_2.12:2.4.0 \  /opt/bitnami/spark/ml/01_logistic_regression.py --fast --sample-size 30000# 2) Decision Treedocker exec spark-master spark-submit --driver-memory 2g --packages io.delta:delta-core_2.12:2.4.0 \  /opt/bitnami/spark/ml/02_decision_tree.py --fast --sample-size 30000# 3) Random Forestdocker exec spark-master spark-submit --driver-memory 2g --packages io.delta:delta-core_2.12:2.4.0 \  /opt/bitnami/spark/ml/03_random_forest.py --fast --sample-size 30000# 4) Gradient Boosted Treesdocker exec spark-master spark-submit --driver-memory 2g --packages io.delta:delta-core_2.12:2.4.0 \  /opt/bitnami/spark/ml/04_gbt.py --fast --sample-size 30000# 5) Naive Bayesdocker exec spark-master spark-submit --driver-memory 2g --packages io.delta:delta-core_2.12:2.4.0 \  /opt/bitnami/spark/ml/05_naive_bayes.py --fast --sample-size 30000```Tüm sonuçlar **MLflow UI** üzerinden izlenebilir: `http://localhost:5000`